# Notebook 12 · ES Arena Prospective Evaluation (v0.2 — legacy submission reconciliation)

**Market Mood Forecasting · pre-registered evaluation framework · reproducible, read-only by default**

> **Research question.** Do the frozen September-30-2026 Logistic Regression probabilities have measurable forecasting value on *future* E-mini S&P 500 futures challenges, relative to fixed baselines and matched external agents?
>
> **Status.** *INTERIM until the planned cohort has 30 officially settled, valid forecasts.* A low early accuracy or confidence is not a verdict on skill. No model retraining, feature engineering, trading, API submission, data buying or strategy selection happens here.

### The three distinct outcomes

1. **Engineering:** Were the forecasts recorded, time-consistent, and auditable?
2. **Forecasting:** How accurate / well-calibrated were their *three probabilities* versus official Arena outcomes and predeclared baselines?
3. **Trading profitability:** **Not measured here**. Notebook 13 will model feasible executions, costs and ROI. Notebook 14 will analyze volatility and geopolitical regimes separately.

**Input provenance.** Forecast JSON files and receipts are generated by `es_arena_daily.py`; daily ES snapshots are generated by Notebook 11. Official settlement outcomes and peer probability vectors must come from **verifiable Arena records**, not from retrospective inferred returns. Do not put API secrets or licensed raw bars in Git.

**v0.2 provenance fix:** October 20 originally used an earlier notebook submission path. A separately evidenced recovered CLI result is included in the 30-forecast cohort, without manufacturing native forecast/receipt JSONs.


## 00 · Run instructions and safeguards

Run from JupyterLab/VS Code with the project's `.venv`, from any working directory inside the project. This notebook locates the repository by `es_arena_daily.py` and `notebooks/` and **only reads project files by default**.

**First run**

1. Check local folders `data/processed/es_arena_live/forecasts/` and `.../receipts/`. Keep the original JSONs immutable.
2. **First import the Oct 20 first-notebook submission** as a *recovered legacy evidence record*, not as a native JSON/receipt. Place the separately provided `legacy_es_20261020_118cefc4.json` in `data/processed/es_arena_research_v0_1/legacy_submissions/` (Git-ignored); preserve your original CLI log/screenshot privately. Run once in read-only mode and check the audit ledger counts **5 accepted** (four native + one recovered), provided the four native records are still available.
3. Use `WRITE_MISSING_TEMPLATES=True` **once** to create blank, Git-ignored CSV templates under `data/processed/es_arena_research_v0_1/`. Then return it to `False`.
4. Populate official outcomes **only after Arena settlement**, with `source_url` pointing to the matching official challenge. Fill peer forecasts only when the original probabilities, exact challenge ID and their timestamp can be verified. Keep a local evidence archive. Do **not** infer official labels from Databento.
5. Optionally fill `training_class_counts.csv` from a *fixed, training-only* pre-October-2026 count; otherwise the historical-prior baseline stays unavailable.
6. Restart Kernel → Run All. When a record is absent/unrecognized, follow the explicit warnings, not a fabricated statistic.

**Legacy limitations:** The October 20 source is a genuine 201 API response transcribed from the original first-notebook console and corroborated by the Arena public page. The *original native file and server timestamp time zone are unavailable*. The recovered record is distinctly labeled, with no invented original receipt, creation-time SHA-256 or native forecast. Model-byte provenance for that specific first submission still needs independent verification from Notebook 10 or historical material. We preserve this limitation and show a sensitivity calculation excluding it.

**Immutability:** no forecast files, receipts, model binaries, original audits, or immutable snapshots are overwritten. No calls to Headline Arena or Databento APIs. Only the optional creation of missing **empty** CSV templates writes a file, with exclusive creation.

**Frozen protocol:** class order `bearish, neutral, bullish`; ES dead-zone ±0.30% of the official opening; primary cohort = **first 30 accepted ES forecasts from 2026-10-20 onward**, in settlement-date order. We record *all* days/forecasts, including missing, invalid and unsuccessful observations; inclusion and exclusion reasons are transparent. Final inference waits for these 30 to settle. If an accepted forecast cannot be audited, do not silently replace it with the 31st.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, hashlib, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CLASSES = ("bearish", "neutral", "bullish")
CLASS_TO_I = {v:i for i,v in enumerate(CLASSES)}
COHORT_START = "2026-10-20"
COHORT_TARGET_N = 30
ORIGINAL_FIRST_CHALLENGE_ID = "118cefc4-96a8-4e47-b617-73af53c67d03"
EXPECTED_CONTRACT = "ESZ6"  # a roll requires documenting/updating the protocol, never silently reinterpreting it
DEAD_ZONE_PCT = 0.3
FROZEN_DEPLOYMENT = "deployment_ab4d5c14423352c7"
FROZEN_MODEL_SHA256 = "5e8a2a9f3ba9a310b340aa09c3ddc823aab2fe9aed728ac87965c2e61a5ae68d"
BOOTSTRAP_SEED = 20261026
WRITE_MISSING_TEMPLATES = False  # ONE-TIME OPT-IN only; never overwrite an existing CSV

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / "es_arena_daily.py").is_file() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside your Market-Mood-Forecasting checkout; repo root not found")

EVIDENCE = PROJECT_ROOT / "data/processed/es_arena_live"
FORECAST_DIR = EVIDENCE / "forecasts"
RECEIPT_DIR = EVIDENCE / "receipts"
SNAPSHOT_DIR = PROJECT_ROOT / "data/processed/es_arena_nb11_v0_1/snapshots"
MODEL_DIR = PROJECT_ROOT / "data/processed/es_arena_nb10_v0_1" / FROZEN_DEPLOYMENT
RESEARCH_DIR = PROJECT_ROOT / "data/processed/es_arena_research_v0_1"
LEGACY_DIR = RESEARCH_DIR / "legacy_submissions"
OFFICIAL_CSV = RESEARCH_DIR / "arena_official_settlements.csv"
PEERS_CSV = RESEARCH_DIR / "arena_peer_forecasts.csv"
TRAIN_COUNTS_CSV = RESEARCH_DIR / "training_class_counts.csv"
SCHEDULE_CSV = RESEARCH_DIR / "evaluation_calendar.csv"

print("PROJECT:", PROJECT_ROOT)
print("MODE:", "OPTIONAL EMPTY TEMPLATE CREATION" if WRITE_MISSING_TEMPLATES else "READ-ONLY")
print("Protocol: 30 accepted ES forecasts; model unchanged; official Arena settlement is ground truth")

## 01 · Protocol fingerprint, environment and provenance

The model identity, training cutoff and source deployment are **historical constants**, not values to optimize after observing outcomes. Compare the local model SHA-256 and its manifest. The run **does not load or fit the estimator**; its saved probability vectors are the source of truth.

*Never change this notebook's threshold, baseline definitions or sample-selection rule based on interim scores. Changes, if any, must be versioned and disclosed prospectively.*

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for part in iter(lambda:f.read(1024*1024),b""):
            h.update(part)
    return h.hexdigest()

def read_json(path):
    with open(path,"r",encoding="utf-8-sig") as f:
        obj=json.load(f)
    if not isinstance(obj,dict):
        raise ValueError(f"Expected JSON object: {path.name}")
    return obj

model_file = MODEL_DIR / "es_daily_lr_deployment_v0_1.joblib"
manifest_file = MODEL_DIR / "es_daily_lr_deployment_v0_1_manifest.json"
metadata_file = MODEL_DIR / "es_daily_lr_deployment_v0_1_metadata.json"
if all(p.is_file() for p in [model_file,manifest_file,metadata_file]):
    manifest=read_json(manifest_file)
    identity=read_json(metadata_file).get("deployment_identity",{})
    assert sha256_file(model_file)==FROZEN_MODEL_SHA256, "Unexpected model bytes: STOP"
    assert manifest.get("model_sha256")==FROZEN_MODEL_SHA256, "Manifest model identity mismatch: STOP"
    assert identity.get("train_through")=="2026-09-30", "Training cutoff mismatch: STOP"
    assert identity.get("class_order")==list(CLASSES), "Class order mismatch: STOP"
    print("FROZEN DEPLOYMENT: VERIFIED")
    print("Model SHA-256:",FROZEN_MODEL_SHA256)
    print("Training cutoff:",identity["train_through"])
else:
    warnings.warn("Frozen deployment files are missing on this machine; provenance NOT verified. Do not publish final results from this run.")

print("pandas:",pd.__version__,"numpy:",np.__version__)
print("UTC analysis time:",datetime.now(timezone.utc).isoformat())

## 02 · Input contracts (private CSV templates)

**Arena official settlements** — `arena_official_settlements.csv`: 1 row per challenge, with official `official_label` (`bearish/neutral/bullish`), `settlement_date` (YYYY-MM-DD), `challenge_id`, `asset` = `ES`, `contract_symbol` (typically `ESZ6`), `source_url` (official Arena challenge URL). Optional: `settlement_return_pct` in **percent units** (e.g. `-0.438` means −0.438%), `official_deadline_utc`, `official_settled_at_utc`, `arena_score`, `verified_by`, `verified_at_utc`. **No official result = no settled metric.** Do not use an empty `Outcome —` as an outcome.

**Peer records** — `arena_peer_forecasts.csv`: one row per `challenge_id` × `agent_id`, with exact three probability columns `p_bearish,p_neutral,p_bullish`, `submitted_at_utc`, `source_url` and optionally `agent_name`, `counts_for_score`. Include only publicly accessible and permitted records, never assume leaderboard summaries represent matched predictions.

**Training counts** — `training_class_counts.csv`: exactly three rows with `class_label,count`, obtained from *training-period only* observations and documented in `source_description`; absent by default because the training labels were not supplied with this notebook. We never estimate historical priors from the evaluation outcomes.

**Calendar** — `evaluation_calendar.csv`: optionally record `settlement_date,status,notes` for closed, missed or unoffered challenges, so non-submissions are visible. Accepted but unresolved records are shown separately.

**Recovered first submission:** A JSON record in `legacy_submissions/` preserves the original HTTP 201 response, prediction ID, exact probabilities and public challenge source. The record must be explicitly marked as a transcription of contemporaneous console evidence, **never** as the original receipt. A separate official-settlements row (with verifiable Arena source URL) supplies its actual bullish outcome.

Blank templates do not constitute observations. Native forecast/receipt JSON schemas may evolve; an unrecognized format is surfaced for adapter review, not silently guessed.

**Git safety prerequisite:** before enabling `WRITE_MISSING_TEMPLATES`, ensure `.gitignore` contains `data/processed/es_arena_research_v0_1/` (or an equivalent broader ignore rule). The template-creation cell checks Git exclusion and refuses to write otherwise.


**Seed row rule:** If you have independently verified the October 20 official result, you may place the separately supplied one-row, sourced CSV at `arena_official_settlements.csv` before enabling `WRITE_MISSING_TEMPLATES=True`; the creation step will preserve it. Never replace an existing, populated settlements CSV with a one-row seed.


In [ ]:
TEMPLATES = {
    OFFICIAL_CSV: ["challenge_id","settlement_date","asset","contract_symbol","official_label","settlement_return_pct","official_deadline_utc","official_settled_at_utc","arena_score","source_url","verified_by","verified_at_utc"],
    PEERS_CSV: ["challenge_id","agent_id","agent_name","p_bearish","p_neutral","p_bullish","submitted_at_utc","counts_for_score","source_url"],
    TRAIN_COUNTS_CSV: ["class_label","count","source_description"],
    SCHEDULE_CSV: ["settlement_date","status","notes"]
}
if WRITE_MISSING_TEMPLATES:
    import subprocess
    assert (PROJECT_ROOT / ".gitignore").exists(), "Review .gitignore first"
    for path in TEMPLATES:
        check=subprocess.run(["git","-C",str(PROJECT_ROOT),"check-ignore","-q",str(path)],capture_output=True)
        if check.returncode!=0:
            raise RuntimeError(f"STOP: {path} is not Git-ignored. Add 'data/processed/es_arena_research_v0_1/' to .gitignore before creating input templates")
    RESEARCH_DIR.mkdir(parents=True,exist_ok=True)
    for path, cols in TEMPLATES.items():
        if path.exists():
            print("PRESERVED EXISTING:", path.name)
        else:
            with path.open("x",encoding="utf-8",newline="") as f:
                f.write(",".join(cols)+"\n")
            print("CREATED EMPTY TEMPLATE:",path)
else:
    for path in TEMPLATES:
        print("PRESENT" if path.exists() else "NOT PROVIDED",path.name)

def load_optional_csv(path):
    if not path.exists():
        return pd.DataFrame(columns=TEMPLATES[path])
    df=pd.read_csv(path, dtype=str,keep_default_na=False)
    missing=set(TEMPLATES[path]).difference(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing required headers: {sorted(missing)}")
    return df

arena_raw=load_optional_csv(OFFICIAL_CSV)
peer_raw=load_optional_csv(PEERS_CSV)
train_raw=load_optional_csv(TRAIN_COUNTS_CSV)
calendar_raw=load_optional_csv(SCHEDULE_CSV)
print(f"Human-verified official rows: {len(arena_raw)} | peer rows: {len(peer_raw)} | calendar rows: {len(calendar_raw)}")

## 03 · Read native saved forecasts (never recompute probabilities)

The adapter accepts the three probabilities only when explicitly named; it **does not derive them from market returns**. Accepted examples: `probabilities`, `class_probabilities`, `prediction.probabilities`, `payload.probabilities`, `forecast.probabilities` or an object with explicitly named `bearish/neutral/bullish` keys. If your saved schema differs, change only the **read-only adapter**, not the source JSON, and document the adapter revision.

Forecast filenames generated by our CLI contain `forecast_YYYYMMDD_<challenge-uuid>.json`; the filename is cross-checked with any identifiers inside the JSON. Duplicate stored forecasts for a challenge are an integrity failure, not a reason to choose the favorable one.

In [ ]:
def at_path(obj, path):
    cur=obj
    for key in path.split("."):
        if not isinstance(cur,dict) or key not in cur:
            return None
        cur=cur[key]
    return cur

def candidates_at_paths(obj, paths):
    return [(path,at_path(obj,path)) for path in paths if at_path(obj,path) is not None]

def unique_value(obj, paths, label):
    values=candidates_at_paths(obj,paths)
    if not values: return None
    norm={str(v).strip() for _,v in values}
    if len(norm)>1:
        raise ValueError(f"Conflicting {label} fields: {values}")
    return values[0][1]

PROB_PATHS = ["probabilities", "class_probabilities", "forecast.probabilities",
              "prediction.probabilities", "payload.probabilities", "prediction_payload.probabilities",
              "forecast", "prediction", "payload", "prediction_payload", ""]

def parse_probabilities(obj):
    found=[]
    for path in PROB_PATHS:
        node=at_path(obj,path) if path else obj
        if not isinstance(node,dict): continue
        choices=[tuple(f"p_{c}" for c in CLASSES), CLASSES]
        for keys in choices:
            if all(k in node for k in keys):
                vec=np.array([float(node[k]) for k in keys],dtype=float)
                found.append((path or "root",vec))
    if not found:
        raise ValueError("No explicit 3-class probability keys found; inspect JSON structure and extend PROB_PATHS")
    first=found[0][1]
    if any(not np.allclose(first,x[1],rtol=0,atol=1e-10) for x in found[1:]):
        raise ValueError("Conflicting probability vectors inside one artifact")
    if not np.isfinite(first).all() or (first<0).any() or (first>1).any() or abs(first.sum()-1)>1e-6:
        raise ValueError(f"Invalid probability vector: {first}")
    return first

FORECAST_NAME=re.compile(r"^forecast_(\d{8})_([0-9a-fA-F-]{36})\.json$")

def load_forecast_file(path):
    obj=read_json(path)
    match=FORECAST_NAME.fullmatch(path.name)
    if not match:
        raise ValueError("Forecast name does not follow immutable expected CLI convention")
    date_iso=f"{match[1][:4]}-{match[1][4:6]}-{match[1][6:8]}"
    cid=match[2].lower()
    stored_id=unique_value(obj,["challenge_id","challenge.id","challenge_uuid","prediction.challenge_id",
                               "forecast.challenge_id","payload.challenge_id"],"challenge ID")
    if stored_id is not None and str(stored_id).lower()!=cid:
        raise ValueError("Filename challenge ID differs from JSON challenge ID")
    stored_date=unique_value(obj,["settlement_date","target_date","session_date",
                                 "date","forecast_date","challenge.session_date"],"target date")
    if stored_date is not None and str(stored_date)[:10]!=date_iso:
        raise ValueError("Filename date differs from JSON target date")
    contract=unique_value(obj,["contract_symbol","contract","raw_symbol","challenge.contract_symbol"],"contract")
    if contract is not None and str(contract)!=EXPECTED_CONTRACT:
        raise ValueError(f"Unexpected contract {contract}; review roll protocol")
    p=parse_probabilities(obj)
    submission_time=unique_value(obj,["created_at_utc","created_at","forecast_created_at_utc",
                                     "prepared_at_utc","timestamp_utc","timestamp"],"forecast timestamp")
    source_date=unique_value(obj,["last_session","last_session_date","source_date",
                                  "source_session_date","latest_session_date"],"latest source session")
    return dict(challenge_id=cid,settlement_date=date_iso,contract_symbol=str(contract or EXPECTED_CONTRACT),
                p_bearish=p[0],p_neutral=p[1],p_bullish=p[2],
                predicted_label=CLASSES[int(np.argmax(p))],confidence=float(p.max()),
                forecast_created_at_utc=str(submission_time or ""),last_session=str(source_date or ""),
                forecast_file=path.name,forecast_sha256=sha256_file(path))

forecast_paths=sorted(FORECAST_DIR.glob("forecast_*.json")) if FORECAST_DIR.is_dir() else []
forecast_rows=[]; forecast_errors=[]
for path in forecast_paths:
    try: forecast_rows.append(load_forecast_file(path))
    except Exception as e: forecast_errors.append((path.name,str(e)))
if forecast_errors:
    print("UNRECOGNIZED / INVALID FORECAST JSONS — investigate before final publication:")
    display(pd.DataFrame(forecast_errors,columns=["file","problem"]))
    print("For schema diagnosis, inspect field NAMES ONLY from your own JSON, not credentials:")
    for filename,_ in forecast_errors[:5]:
        try: print(filename, "root keys:",list(read_json(FORECAST_DIR / filename).keys()))
        except Exception: pass
forecasts=pd.DataFrame(forecast_rows)
print("Forecast files found:",len(forecast_paths),"| parsed:",len(forecasts),"| errors:",len(forecast_errors))
if not forecasts.empty:
    assert not forecasts.challenge_id.duplicated().any(), "Duplicate immutable forecasts per challenge: STOP"
    display(forecasts[["settlement_date","challenge_id","predicted_label","confidence","forecast_file"]].sort_values("settlement_date"))

## 04 · Audit official submission receipts and deadlines

For **confirmed acceptance**, require an actual receipt, `HTTP 201` and an explicit `counts_for_score: true` when those fields are exposed. Unknown receipt formats are **not** silently accepted. Timestamp checks use only UTC-aware values; missing timestamps/deadlines are recorded as **UNKNOWN**, not falsely labeled timely.

A receipt may be missing while a platform submission actually succeeded; reconcile against Arena evidence rather than retrying a forecast or manufacturing a receipt.

In [ ]:
def first_nested(obj, paths):
    for path in paths:
        v=at_path(obj,path)
        if v is not None: return v
    return None

def parse_utc(v):
    if v is None or (isinstance(v,str) and not v.strip()): return pd.NaT
    dt=pd.to_datetime(v,utc=True,errors="coerce")
    if isinstance(dt,pd.DatetimeIndex):
        return pd.NaT
    return dt

def bool_is_true(v):
    return v is True or (isinstance(v,str) and v.strip().lower()=="true")

RECEIPT_NAME=re.compile(r"^receipt_([0-9a-fA-F-]{36})\.json$")

def load_receipt_file(path):
    o=read_json(path)
    m=RECEIPT_NAME.fullmatch(path.name)
    if not m: raise ValueError("Receipt filename not recognized")
    cid=m[1].lower()
    stored=first_nested(o,["challenge_id","response.challenge_id","request.challenge_id",
                           "prediction.challenge_id","receipt.challenge_id"])
    if stored is not None and str(stored).lower()!=cid:
        raise ValueError("Receipt challenge ID mismatch")
    status=first_nested(o,["http_status","http_status_code","status_code","response.status_code",
                           "submission.http_status","receipt.http_status","server_response.status_code",
                           "arena_response.status_code","result.http_status"])
    accepted=first_nested(o,["counts_for_score","response.counts_for_score",
                             "response.prediction.counts_for_score","receipt.counts_for_score",
                             "server_response.counts_for_score","arena_response.counts_for_score",
                             "prediction.counts_for_score","submission.counts_for_score"])
    submitted=first_nested(o,["submitted_at_utc","submitted_at","response.submitted_at_utc",
                              "response.created_at","prediction.created_at","timestamp_utc"])
    model_code=str(status) if status is not None else ""
    confirmed=(model_code=="201") and bool_is_true(accepted)
    return dict(challenge_id=cid, receipt_file=path.name,receipt_sha256=sha256_file(path),
                receipt_http_status=model_code, receipt_counts_for_score=str(accepted),
                accepted_by_receipt=bool(confirmed),submitted_at_utc=str(submitted or ""))

receipt_paths=sorted(RECEIPT_DIR.glob("receipt_*.json")) if RECEIPT_DIR.is_dir() else []
receipt_rows=[];receipt_errors=[]
for path in receipt_paths:
    try: receipt_rows.append(load_receipt_file(path))
    except Exception as e: receipt_errors.append((path.name,str(e)))
receipts=pd.DataFrame(receipt_rows)
if receipt_errors: display(pd.DataFrame(receipt_errors,columns=["receipt_file","problem"]))
print("Receipt files:",len(receipt_paths),"| accepted by strict adapter:",sum(x["accepted_by_receipt"] for x in receipt_rows),"| parse errors:",len(receipt_errors))
if not receipts.empty:
    assert not receipts.challenge_id.duplicated().any(),"Duplicate receipts per challenge: STOP"
    if (~receipts.accepted_by_receipt).any():
        print("WARNING: receipt schema unconfirmed for:")
        display(receipts.loc[~receipts.accepted_by_receipt,["challenge_id","receipt_http_status","receipt_counts_for_score"]])
        print("Unverified receipt ROOT KEYS (no values/secrets printed):")
        for name in receipts.loc[~receipts.accepted_by_receipt,"receipt_file"].head(5):
            try: print(name, list(read_json(RECEIPT_DIR / name).keys()))
            except Exception: pass

# IMPORTANT: recover independently verified first-notebook submissions without fabricating
# a modern immutable forecast or a modern receipt. The original evidence file is never edited.
from urllib.parse import urlparse
import uuid

def load_recovered_legacy(path):
    obj=read_json(path)
    if obj.get("evidence_kind")!="recovered_original_cli_transcription_v1":
        raise ValueError("Legacy evidence must be explicitly marked as reconstructed from original console output")
    cid=str(obj.get("challenge_id", "")).lower()
    if not cid or str(uuid.UUID(cid))!=cid:
        raise ValueError("Legacy challenge UUID malformed")
    result=obj.get("server_response")
    if not isinstance(result,dict): raise ValueError("Missing preserved original server response")
    if str(result.get("challenge_id", "")).lower()!=cid: raise ValueError("Server/challenge ID conflict")
    if str(result.get("prediction_id", "")).strip()=="": raise ValueError("No server prediction ID")
    if str(obj.get("http_status"))!="201" or result.get("counts_for_score") is not True:
        raise ValueError("Legacy record not confirmed accepted with HTTP 201 and scored=True")
    date=str(obj.get("settlement_date", ""))
    if not re.fullmatch(r"20\d{2}-\d{2}-\d{2}",date): raise ValueError("Bad legacy date")
    if str(obj.get("contract_symbol", ""))!=EXPECTED_CONTRACT: raise ValueError("Unexpected legacy contract")
    url=str(obj.get("official_challenge_url", ""))
    parsed=urlparse(url)
    if parsed.scheme!="https" or parsed.hostname not in {"headlinearena.com","www.headlinearena.com"} or cid not in parsed.path:
        raise ValueError("Recovered record needs exact Arena public challenge URL")
    if not str(obj.get("source_note", "")).strip(): raise ValueError("Explicit legacy provenance note required")
    probabilities=parse_probabilities(result)
    direction=CLASSES[int(np.argmax(probabilities))]
    if str(result.get("direction"))!=direction: raise ValueError("Direction inconsistent with recorded probabilities")
    if not np.isclose(float(result.get("confidence")),probabilities.max(),atol=1e-10,rtol=0):
        raise ValueError("Legacy confidence differs from probabilities")
    # The API server's raw timestamp has no UTC offset. NEVER invent one.
    raw_time=str(result.get("created_at", ""))
    if not raw_time: raise ValueError("Missing original raw server creation timestamp")
    return dict(challenge_id=cid,settlement_date=date,contract_symbol=EXPECTED_CONTRACT,
                p_bearish=float(probabilities[0]),p_neutral=float(probabilities[1]),p_bullish=float(probabilities[2]),
                predicted_label=direction,confidence=float(probabilities.max()),
                forecast_created_at_utc="",last_session="",forecast_file="",forecast_sha256="",
                receipt_file="",receipt_sha256="",receipt_http_status="201",receipt_counts_for_score="True",
                accepted_by_receipt=False,accepted_by_external_evidence=True,
                submitted_at_utc="",legacy_server_created_at_raw=raw_time,
                evidence_type="RECOVERED_CLI_201_PLUS_OFFICIAL_PAGE",evidence_file=path.name,
                legacy_file_sha256=sha256_file(path),native_artifacts_present=False,
                original_model_sha_verified=False)

legacy_paths=sorted(LEGACY_DIR.glob("legacy_*.json")) if LEGACY_DIR.is_dir() else []
legacy_rows=[];legacy_errors=[]
for path in legacy_paths:
    try: legacy_rows.append(load_recovered_legacy(path))
    except Exception as exc: legacy_errors.append((path.name,str(exc)))
if legacy_errors:
    display(pd.DataFrame(legacy_errors,columns=["legacy_file","error"]))
    raise ValueError("Legacy evidence failed validation — STOP; correct it before continuing")
if legacy_rows:
    print("RECOVERED ORIGINAL CONSOLE EVIDENCE (NOT ORIGINAL JSON OR RECEIPT):",len(legacy_rows))
    display(pd.DataFrame(legacy_rows)[["settlement_date","challenge_id","predicted_label","confidence","evidence_type"]])
else:
    print("NO LEGACY FIRST-NOTEBOOK EVIDENCE FOUND — cohort may be missing October 20")

native = forecasts.merge(receipts,on="challenge_id",how="left",validate="one_to_one") if (not forecasts.empty and not receipts.empty) else forecasts.copy()
if not native.empty:
    if "accepted_by_receipt" not in native: native["accepted_by_receipt"]=False
    native["accepted_by_receipt"]=native["accepted_by_receipt"].fillna(False).astype(bool)
    native["accepted_by_external_evidence"]=False
    native["evidence_type"]="NATIVE_FORECAST_PLUS_RECEIPT"
    native["native_artifacts_present"]=True
    native["original_model_sha_verified"]=True  # model SHA checked locally; inference source still audited separately
    native["legacy_server_created_at_raw"]=""
    native["evidence_file"]=""
    native["legacy_file_sha256"]=""
if legacy_rows and not native.empty:
    duplicates=set(native.challenge_id) & {r["challenge_id"] for r in legacy_rows}
    if duplicates:
        raise ValueError(f"Legacy and native forecasts overlap; reconcile before evaluation: {sorted(duplicates)}")
frames=[f for f in [native, pd.DataFrame(legacy_rows)] if not f.empty]
ledger=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame(columns=["challenge_id","settlement_date","predicted_label","confidence","accepted_by_receipt","accepted_by_external_evidence"])
if not ledger.empty:
    assert not ledger.challenge_id.duplicated().any(),"Duplicate challenge evidence: STOP"
    for flag in ["accepted_by_receipt","accepted_by_external_evidence"]:
        ledger[flag]=ledger[flag].fillna(False).astype(bool)
    ledger["accepted_with_evidence"]=ledger.accepted_by_receipt | ledger.accepted_by_external_evidence
    ledger=ledger.sort_values(["settlement_date","challenge_id"]).reset_index(drop=True)
    ledger["accepted_rank"]=np.nan
    accepted_pos=ledger.index[ledger.accepted_with_evidence & (ledger.settlement_date>=COHORT_START)]
    ledger.loc[accepted_pos,"accepted_rank"]=np.arange(1,len(accepted_pos)+1)
    ledger["in_first_30_accepted"]=ledger.accepted_rank.between(1,COHORT_TARGET_N)
    ledger["source_date_before_target"]=ledger.apply(lambda r: (not str(r.get("last_session", "") or "").strip()) or (str(r.last_session)[:10]<str(r.settlement_date)),axis=1)
    if not ledger.source_date_before_target.all(): raise ValueError("Source session is not prior to target date: STOP")
else:
    ledger["accepted_with_evidence"]=pd.Series(dtype=bool)
    ledger["in_first_30_accepted"]=pd.Series(dtype=bool)
print("Accepted native forecasts:",sum(x.get("accepted_by_receipt",False) for x in (native.to_dict('records') if not native.empty else [])))
print("Accepted recovered legacy forecasts:",len(legacy_rows))
print("TOTAL accepted records in cohort ledger:",int(ledger.accepted_with_evidence.sum()) if not ledger.empty else 0)
if ORIGINAL_FIRST_CHALLENGE_ID not in (set(ledger.challenge_id) if not ledger.empty else set()):
    print("COHORT INTEGRITY WARNING: original 2026-10-20 accepted submission missing. Final completion is BLOCKED until reconciled.")
if not ledger.empty:
    display(ledger[["settlement_date","challenge_id","predicted_label","accepted_by_receipt","accepted_by_external_evidence","accepted_rank","in_first_30_accepted"]])

## 05 · Official Arena ground truth — validate and join

**Do not equate the local Databento label with Arena's official classification.** Official results are entered from the challenge page, with a source URL. The evaluation joins by the exact immutable `challenge_id` and checks date/contract consistency. Duplicate outcomes and contradictory labels are errors.

*Source discipline:* URLs must identify `headlinearena.com/prediction/<challenge_id>` or another verified official Arena record; never fabricate a result from the public summary percentage, which might have rounding or settlement-price sampling differences.

In [ ]:
def validate_official(df):
    if df.empty: return df.copy()
    d=df.copy()
    assert d.challenge_id.ne("").all() and not d.challenge_id.duplicated().any(), "Blank or duplicate challenge IDs in official CSV"
    assert d.official_label.isin(CLASSES).all(),"Invalid official labels"
    assert d.asset.eq("ES").all(),"Official CSV must be ES-only"
    assert d.settlement_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(),"Bad settlement dates"
    assert d.source_url.ne("").all(),"Official source URL required"
    from urllib.parse import urlparse
    assert d.apply(lambda r: (urlparse(r.source_url).scheme=="https") and
                   (urlparse(r.source_url).hostname in {"headlinearena.com","www.headlinearena.com"}) and
                   (r.challenge_id.lower() in urlparse(r.source_url).path.lower()),axis=1).all(),"Use an HTTPS Arena source URL for this exact challenge"
    assert d.contract_symbol.ne("").all(),"Contract required to verify same underlying"
    d["challenge_id"]=d.challenge_id.str.lower().str.strip()
    d["settlement_return_pct"]=pd.to_numeric(d.settlement_return_pct.replace("",np.nan),errors="coerce")
    d["arena_score"]=pd.to_numeric(d.arena_score.replace("",np.nan),errors="coerce")
    return d

arena=validate_official(arena_raw)
if ledger.empty:
    full=ledger.copy()
else:
    full=ledger.merge(arena,on="challenge_id",how="left",suffixes=("","_official"),validate="one_to_one") if not arena.empty else ledger.copy()
    if "official_label" not in full: full["official_label"]=np.nan
    if "settlement_date_official" in full:
        conflict=full.settlement_date_official.notna() & (full.settlement_date_official!=full.settlement_date)
        if conflict.any(): raise ValueError("Challenge date in official CSV mismatches forecast: STOP")
    if "contract_symbol_official" in full:
        conflict=full.contract_symbol_official.notna() & (full.contract_symbol_official!=full.contract_symbol)
        if conflict.any(): raise ValueError("Official contract differs: STOP")
    full["settled_confirmed"]=full.official_label.isin(CLASSES)
    full["eligible_for_metrics"]=full.in_first_30_accepted.fillna(False) & full.settled_confirmed
    if "official_deadline_utc" in full:
        full["submission_deadline_status"]=[
            "VERIFIED" if pd.notna(parse_utc(s)) and pd.notna(parse_utc(d)) and parse_utc(s)<=parse_utc(d)
            else ("LATE" if pd.notna(parse_utc(s)) and pd.notna(parse_utc(d)) and parse_utc(s)>parse_utc(d) else "UNKNOWN")
            for s,d in zip(full.get("submitted_at_utc",[""]*len(full)),full.official_deadline_utc)
        ]
        # A legacy record can prove server-side scored acceptance without
        # asserting that its timestamp lacking a UTC offset is a verified UTC time.
        recovered=full.accepted_by_external_evidence.fillna(False).astype(bool)
        full.loc[recovered & full.submission_deadline_status.eq("UNKNOWN"),
                 "submission_deadline_status"]="SERVER_ACCEPTED_SCORING_ELIGIBLE_UNZONED_TIME"
        if (full.submission_deadline_status=="LATE").any():
            raise ValueError("Documented late submission detected: STOP before evaluating")
    else:
        full["submission_deadline_status"]="UNKNOWN"
        recovered=full.accepted_by_external_evidence.fillna(False).astype(bool)
        full.loc[recovered,"submission_deadline_status"]="SERVER_ACCEPTED_SCORING_ELIGIBLE_UNZONED_TIME"

print("Official verified outcome rows:",len(arena))
print("First-30 cohort, officially settled with accepted evidence:",int(full.eligible_for_metrics.sum()) if not full.empty else 0)
print("Preliminary:", (len(full)==0 or full.eligible_for_metrics.sum()<COHORT_TARGET_N))
if not full.empty:
    display(full[["settlement_date","challenge_id","predicted_label","official_label","accepted_by_receipt","eligible_for_metrics","submission_deadline_status"]])
if not full.empty and full.eligible_for_metrics.sum() < COHORT_TARGET_N:
    print("PRELIMINARY ONLY: the first 30 accepted challenges have not all settled and passed validation.")

## 06 · Metric definitions and self-tests

**Primary:** accuracy of the maximum-probability class; *unscaled* three-class Brier score `mean(sum_k (p_k - 1[y=k])²)`, lower is better (range 0–2). This may differ numerically from an Arena-normalized score. **Secondary:** log loss (epsilon = `1e-15`), confidence, per-class precision/recall, macro-F1, and multiclass confusion matrix. Use official Arena class, not locally inferred class.

Self-tests below use **obviously artificial arrays** to catch arithmetic mistakes. They are never presented as model performance.

In [ ]:
def arrays_from_rows(frame):
    p=frame[[f"p_{c}" for c in CLASSES]].to_numpy(dtype=float)
    labels=frame.official_label.astype(str).to_numpy()
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any() or not np.allclose(p.sum(axis=1),1,atol=1e-6):
        raise ValueError("Invalid model probability vectors")
    if not set(labels).issubset(CLASSES): raise ValueError("Invalid official labels")
    return p,np.array([CLASS_TO_I[x] for x in labels],dtype=int)

def score_probabilities(p,y):
    p=np.asarray(p,dtype=float)
    y=np.asarray(y,dtype=int)
    assert p.ndim==2 and p.shape[1]==3 and len(y)==len(p) and len(y)>0
    assert np.allclose(p.sum(axis=1),1,atol=1e-6)
    one=np.eye(3)[y]
    pred=p.argmax(axis=1)
    return dict(n=len(y),accuracy=float(np.mean(pred==y)),
        avg_confidence=float(p.max(axis=1).mean()),
        brier=float(np.sum((p-one)**2,axis=1).mean()),
        log_loss=float(-np.log(np.clip(p[np.arange(len(y)),y],1e-15,1)).mean()))

# Unit tests: perfect, uniform, degenerate and general three-class distributions.
P=np.eye(3)
Y=np.array([0,1,2])
assert score_probabilities(P,Y)["brier"]==0 and score_probabilities(P,Y)["accuracy"]==1
U=np.full((3,3),1/3)
assert np.isclose(score_probabilities(U,Y)["brier"],2/3)
assert np.isclose(score_probabilities(U,Y)["log_loss"],np.log(3))
assert np.isclose(score_probabilities(np.array([[1.,0.,0.]]),np.array([2]))["brier"],2)
print("METRIC UNIT TESTS: PASS (test examples are NOT research results)")

## 07 · Performance on the currently settled cohort

Interim results are shown only when **confirmed accepted forecasts have corresponding official settled outcomes**. No synthetic scores appear here. For missing outcomes or ambiguous receipts, inspect the input-quality report above. Avoid treating four settled forecasts as a statistically useful skill estimate.

In [ ]:
eval_df = full.loc[full.eligible_for_metrics].copy() if not full.empty else pd.DataFrame()
MODEL_METRICS=None
if eval_df.empty:
    print("NO ELIGIBLE OFFICIALLY SETTLED RECORDS YET — no real performance statistics calculated.")
else:
    p,y=arrays_from_rows(eval_df)
    MODEL_METRICS=score_probabilities(p,y)
    display(pd.DataFrame([{"metric":k,"value":v} for k,v in MODEL_METRICS.items()]))
    cm=pd.crosstab(pd.Categorical(eval_df.official_label,categories=CLASSES),
                   pd.Categorical(eval_df.predicted_label,categories=CLASSES),dropna=False)
    cm.index.name="Official outcome";cm.columns.name="Model top class"
    display(cm)
    class_stats=[]
    for k,c in enumerate(CLASSES):
        actual=(y==k); chosen=(p.argmax(axis=1)==k)
        tp=int((actual&chosen).sum())
        class_stats.append({"class":c,"support":int(actual.sum()),"precision":tp/max(int(chosen.sum()),1) if chosen.sum() else np.nan,
                            "recall":tp/max(int(actual.sum()),1) if actual.sum() else np.nan})
    display(pd.DataFrame(class_stats))
    
    # Audit table: all officially settled accepted forecasts.
    daily_audit = eval_df[
        [
            "settlement_date",
            "challenge_id",
            "predicted_label",
            "official_label",
            "arena_score",
        ]
    ].copy()

    daily_audit["correct"] = (
        daily_audit["predicted_label"]
        == daily_audit["official_label"]
    )

    print("OFFICIAL SETTLED FORECASTS — CHRONOLOGICAL AUDIT")
    display(
        daily_audit.sort_values("settlement_date")
                   .reset_index(drop=True)
    )

    print(f"INTERIM: {len(eval_df)}/{COHORT_TARGET_N} confirmed settled; do not claim this is final model skill.")
    if "accepted_by_external_evidence" in eval_df.columns and eval_df.accepted_by_external_evidence.fillna(False).any():
        native_only=eval_df.loc[~eval_df.accepted_by_external_evidence.fillna(False)].copy()
        if native_only.empty: print("Legacy sensitivity: no native settled observations yet")
        else:
            q,z=arrays_from_rows(native_only)
            print("Sensitivity excluding recovered legacy submission (NOT the primary cohort):")
            display(pd.DataFrame([score_probabilities(q,z)]))

## 08 · Fixed, non-leaking baselines

**Uniform:** 1/3 for each category, always available. **Training-only prior:** observed frequencies in the original training portion, not the prospective test sample, only when supplied with credible provenance. **Always-bullish / neutral / bearish:** deterministic top-class references (not probability-perfect models); report accuracy only unless pre-smoothing is declared prospectively.

Do not use the evaluation cohort's class prevalence as a predictive baseline; that would leak the answers. Any new rule introduced after seeing current results must be labeled exploratory.

In [ ]:
prior=None
if not train_raw.empty:
    d=train_raw.copy()
    assert len(d)==3 and set(d.class_label)==set(CLASSES) and not d.class_label.duplicated().any(),"Training prior needs precisely 3 unique classes"
    counts=np.array([float(d.loc[d.class_label==c,"count"].iloc[0]) for c in CLASSES])
    assert np.isfinite(counts).all() and (counts>=0).all() and counts.sum()>0
    assert d.source_description.ne("").all(),"Training-only provenance required"
    prior=counts/counts.sum()
    print("Training-only priors, frozen provenance:",dict(zip(CLASSES,prior)))
else: print("TRAINING-ONLY PRIOR: unavailable; intentionally not estimated from prospective outcomes.")

comparison=[]
if not eval_df.empty:
    p,y=arrays_from_rows(eval_df)
    models={"Frozen Logistic Regression":p,"Uniform 1/3":np.full_like(p,1/3)}
    if prior is not None: models["Frozen training-only class prior"]=np.tile(prior,(len(y),1))
    
    for name, q in models.items():
        metrics = score_probabilities(q, y)

        # Uniform probabilities have no unique predicted class.
        # Do not report arbitrary argmax tie-breaking as accuracy.
        if name == "Uniform 1/3":
            metrics["accuracy"] = np.nan

        comparison.append({
            "model": name,
            **metrics,
        })

    # Perpetual one-label rules: accuracy only. Do not masquerade as calibrated probabilistic models.
    for i,c in enumerate(CLASSES):
        comparison.append({"model":f"Always {c} (hard-class, accuracy only)","n":len(y),
                           "accuracy":float(np.mean(y==i)),"avg_confidence":np.nan,"brier":np.nan,"log_loss":np.nan})
    comparison_df=pd.DataFrame(comparison)
    display(comparison_df)
    if len(y):
        print("Skill vs uniform Brier (positive = better):",round(1-MODEL_METRICS["brier"]/(2/3),4))
else:
    comparison_df=pd.DataFrame()
    print("BASELINE TABLE: pending official settled outcomes.")

## 09 · Calibration and uncertainty

Confidence means `max(p_bearish, p_neutral, p_bullish)`, **not** probability that the top prediction is always correct. A calibrated ~39% forecast should eventually be correct approximately 39% of the time *across similarly confident predictions*, but four or even 30 observations offer weak evidence.

We show 3 coarse top-label reliability bins (0–⅓, ⅓–⅔, ⅔–1; practically the first is empty), one-vs-rest class calibration and Wilson 95% accuracy interval. Binning is *descriptive*, not a tuning step. Zero-sample bins are shown as missing.

In [ ]:
def wilson_interval(success,total,z=1.959963984540054):
    if total==0:return (np.nan,np.nan)
    p=success/total;den=1+z*z/total
    ctr=(p+z*z/(2*total))/den
    rad=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return max(0,ctr-rad),min(1,ctr+rad)

if eval_df.empty:
    print("CALIBRATION: pending verified settled observations.")
else:
    p,y=arrays_from_rows(eval_df)
    correct=(p.argmax(axis=1)==y)
    conf=p.max(axis=1)
    lo,hi=wilson_interval(int(correct.sum()),len(correct))
    print(f"Accuracy: {correct.mean():.1%} | Wilson 95% interval: [{lo:.1%}, {hi:.1%}] | n={len(correct)}")
    print("Wilson interval assumes effectively independent Bernoulli events; time dependence may widen uncertainty.")
    bins=np.array([0,1/3,2/3,1.0000001])
    ids=np.clip(np.searchsorted(bins,conf,side="right")-1,0,2)
    cal=[]
    for i in range(3):
        m=(ids==i)
        cal.append({"confidence_bin":f"{bins[i]:.2f}–{min(1,bins[i+1]):.2f}","n":int(m.sum()),
                    "mean_confidence":float(conf[m].mean()) if m.any() else np.nan,
                    "observed_accuracy":float(correct[m].mean()) if m.any() else np.nan})
    display(pd.DataFrame(cal))
    onehot=np.eye(3)[y]
    class_cal=[]
    for j,c in enumerate(CLASSES):
        for a,b in [(0,1/3),(1/3,2/3),(2/3,1.0000001)]:
            m=(p[:,j]>=a)&(p[:,j]<b)
            class_cal.append({"class":c,"p_range":f"[{a:.2f},{min(b,1):.2f}]","n":int(m.sum()),
                              "mean_probability":float(p[m,j].mean()) if m.any() else np.nan,
                              "empirical_frequency":float(onehot[m,j].mean()) if m.any() else np.nan})
    display(pd.DataFrame(class_cal))
    fig,ax=plt.subplots(figsize=(6.4,4))
    tab=pd.DataFrame(cal).dropna(subset=["mean_confidence","observed_accuracy"])
    ax.plot([0,1],[0,1],linestyle="--",label="Perfect calibration")
    if not tab.empty: ax.scatter(tab.mean_confidence,tab.observed_accuracy,s=45,label="Observed confidence bins")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Average predicted confidence",ylabel="Fraction correct",title="Top-class calibration — preliminary")
    ax.legend();plt.show()

## 10 · External Headline Arena competitor benchmarking

**Primary peer estimand:** compare the exact same ES `challenge_id` with another agent's **valid, pre-deadline probability vector**, using the *same official outcome*. Report matched `n`, the number of our total eligible observations that have a peer, and paired changes in accuracy and unscaled Brier. An agent with fewer matched forecasts cannot be evaluated on the full cohort; no imputation of absent predictions.

**Secondary:** a public leaderboard rank/score is *context only*, because agents may cover different challenges, assets and dates. We do not treat an all-market leaderboard ranking as a head-to-head ES comparison. Peer identity and data rights must be documented.

In [ ]:
def validate_peers(raw):
    if raw.empty:return raw.copy()
    d=raw.copy()
    assert d.challenge_id.ne("").all() and d.agent_id.ne("").all(),"Missing peer IDs"
    assert not d.duplicated(["challenge_id","agent_id"]).any(),"Duplicate peer rows"
    from urllib.parse import urlparse
    assert d.apply(lambda r: urlparse(r.source_url).scheme=="https" and
                   urlparse(r.source_url).hostname in {"headlinearena.com","www.headlinearena.com"} and
                   r.challenge_id.lower() in urlparse(r.source_url).path.lower(),axis=1).all(),"Peer official HTTPS source must reference same challenge"
    assert d.counts_for_score.str.lower().eq("true").all(),"Peer must be recorded as scoring-eligible; skip unverified peers"
    assert d.submitted_at_utc.ne("").all(),"Peer timestamp required for comparability"
    for c in CLASSES: d[f"p_{c}"]=pd.to_numeric(d[f"p_{c}"],errors="raise")
    prob=d[[f"p_{c}" for c in CLASSES]].to_numpy(dtype=float)
    assert np.isfinite(prob).all() and (prob>=0).all() and (prob<=1).all() and np.allclose(prob.sum(axis=1),1,atol=1e-6),"Invalid peer probabilities"
    d["challenge_id"]=d.challenge_id.str.lower()
    return d

peer=validate_peers(peer_raw)
peer_results=[]
if eval_df.empty or peer.empty:
    print("MATCHED ARENA AGENT COMPARISON: unavailable until peer probabilities and official results are documented.")
else:
    joined=peer.merge(eval_df[["challenge_id","official_label","settlement_date",
                                  "p_bearish","p_neutral","p_bullish"]],
                      on="challenge_id",suffixes=("_peer","_ours"),validate="many_to_one")
    if "official_deadline_utc" in eval_df:
        dl=eval_df[["challenge_id","official_deadline_utc"]].drop_duplicates("challenge_id")
        joined=joined.merge(dl,on="challenge_id",how="left",validate="many_to_one")
        ok=[]
        for a,b in zip(joined.submitted_at_utc,joined.official_deadline_utc):
            ok.append(pd.notna(parse_utc(a)) and pd.notna(parse_utc(b)) and parse_utc(a)<=parse_utc(b))
        joined=joined.loc[ok].copy()
        if len(joined)==0: print("No peers with verified on-time challenge-matched submissions")
    else:
        print("WARNING: No official deadlines in outcomes CSV; peer timestamp validity unverified — comparison excluded.")
        joined=joined.iloc[0:0]
    for agent_id,d in joined.groupby("agent_id",sort=True):
        ours=d[[f"p_{c}_ours" for c in CLASSES]].to_numpy(dtype=float)
        theirs=d[[f"p_{c}_peer" for c in CLASSES]].to_numpy(dtype=float)
        y=np.array([CLASS_TO_I[t] for t in d.official_label])
        a=score_probabilities(ours,y); b=score_probabilities(theirs,y)
        peer_results.append({"agent_id":agent_id,"matched_n":len(d),"coverage_of_our_settled_pct":100*len(d)/len(eval_df),
                             "our_accuracy_matched":a["accuracy"],"peer_accuracy_matched":b["accuracy"],
                             "peer_minus_our_accuracy":b["accuracy"]-a["accuracy"],
                             "our_brier_matched":a["brier"],"peer_brier_matched":b["brier"],
                             "peer_minus_our_brier":b["brier"]-a["brier"]})
    if peer_results: display(pd.DataFrame(peer_results).sort_values(["matched_n","peer_brier_matched"],ascending=[False,True]))
    else: print("No comparable matched peers; this is NOT a competitor-performance conclusion.")

## 11 · Local Databento ↔ Arena settlement-target reconciliation

This is a **label-alignment audit**, not the official scoring source. The local ES session uses Databento's 18:00–17:00 ET minute-bar construction. Arena may sample a slightly different settlement price. We compare returns (percent units) and the implied bullish/neutral/bearish labels for **the matching challenge settlement date**, and flag mismatches or proximity to ±0.3 percentage points.

Overlapping immutable snapshots are cross-checked; conflicting historical records cause a stop. If a snapshot is unavailable, mark local comparison missing—not a zero difference. This section does not edit old labels or model features.

In [ ]:
def class_from_pct(pct):
    if pd.isna(pct): return None
    if pct>DEAD_ZONE_PCT:return "bullish"
    if pct< -DEAD_ZONE_PCT:return "bearish"
    return "neutral"

local_rows=[]
for path in sorted(SNAPSHOT_DIR.glob("es_sessions_*.csv")) if SNAPSHOT_DIR.is_dir() else []:
    try:
        d=pd.read_csv(path)
        assert {"session_date","session_return","open","close","raw_symbol","coverage_reviewed"}.issubset(d.columns)
        for row in d.itertuples(index=False):
            local_rows.append({"session_date":str(row.session_date),"open":float(row.open),"close":float(row.close),
                               "local_return_pct":float(row.session_return)*100,"raw_symbol":str(row.raw_symbol),
                               "coverage_reviewed":str(row.coverage_reviewed).lower()=="true",
                               "snapshot_file":path.name})
    except Exception as exc:
        warnings.warn(f"Unable to parse local snapshot {path.name}: {exc}")
local_df=pd.DataFrame(local_rows)
if not local_df.empty:
    for date,group in local_df.groupby("session_date"):
        columns=["open","close","local_return_pct","raw_symbol"]
        if len(group)>1:
            assert all(np.isclose(group[x].astype(float).max(),group[x].astype(float).min(),rtol=0,atol=1e-8)
                       if x!="raw_symbol" else group[x].nunique()==1 for x in columns),f"Historical snapshot drift: {date}"
    local_df=local_df.sort_values("snapshot_file").drop_duplicates("session_date",keep="first")
    local_df["local_label"]=local_df.local_return_pct.map(class_from_pct)
    print("Cross-snapshot overlap validation: PASS; unique sessions:",len(local_df))
else: print("No validated local snapshots available for settlement comparison.")

if not eval_df.empty and not local_df.empty:
    target_check=eval_df.merge(local_df,left_on="settlement_date",right_on="session_date",how="left",validate="many_to_one")
    target_check["label_agrees"]=target_check.official_label==target_check.local_label
    target_check["return_diff_pp"]=np.nan
    if "settlement_return_pct" in target_check:
        target_check["return_diff_pp"]=target_check.local_return_pct-target_check.settlement_return_pct
    target_check["distance_to_threshold_pp"]=(target_check.local_return_pct.abs()-DEAD_ZONE_PCT).abs()
    columns=["settlement_date","challenge_id","official_label","local_label","label_agrees","local_return_pct","return_diff_pp","distance_to_threshold_pp"]
    display(target_check[columns])
    print("Official/local class disagreements (only known local rows):",int((target_check.local_label.notna() & ~target_check.label_agrees).sum()))
else: print("Target reconciliation awaits both official outcomes and local snapshots.")

## 12 · Sensitivity and uncertainty (exploratory only)

**Avoid data dredging:** 30 time-ordered outcomes are too few to confidently detect geopolitical causal effects. Provide accuracy uncertainty and a descriptive volatility proxy only if the necessary **contemporaneous, pre-declared** market data exists. More rigorous geopolitical event attribution belongs in Notebook 14.

We use a **seeded simple nonparametric bootstrap** of the settled forecast observations only as a *descriptive sensitivity interval*; serial dependence means it is not a rigorous market-time-series confidence interval. The Wilson interval above is reported for classification accuracy. Do not use dozens of post-hoc variants and select the winner.

In [ ]:
def iid_bootstrap_intervals(p,y,iterations=3000,seed=BOOTSTRAP_SEED):
    rng=np.random.default_rng(seed)
    n=len(y)
    if n==0: return None
    picks=rng.integers(0,n,size=(iterations,n))
    correct=(p.argmax(axis=1)==y).astype(float)
    ones=np.eye(3)[y]
    row_brier=((p-ones)**2).sum(axis=1)
    return {
        "accuracy_2.5%":float(np.quantile(correct[picks].mean(axis=1),.025)),
        "accuracy_97.5%":float(np.quantile(correct[picks].mean(axis=1),.975)),
        "brier_2.5%":float(np.quantile(row_brier[picks].mean(axis=1),.025)),
        "brier_97.5%":float(np.quantile(row_brier[picks].mean(axis=1),.975)),
    }
if eval_df.empty:
    print("Uncertainty intervals unavailable until officially settled outcomes exist.")
else:
    p,y=arrays_from_rows(eval_df)
    print("Illustrative iid bootstrap intervals, NOT guaranteed time-series confidence intervals:")
    display(pd.DataFrame([iid_bootstrap_intervals(p,y)]))
    if len(y)<30: print("EARLY SAMPLE — do not interpret these intervals as evidence of model skill")

## 13 · Integrity / completeness report and eventual publication gate

**Primary experiment is complete only when the first 30 eligible accepted ES forecasts are represented and settled.** If any are missing, unverified, late, duplicated or unresolved, the report must show that status; it cannot silently advance to newer convenient forecasts.

Publishing rules: distinguish `API success` vs `forecast accuracy` vs `ROI`; use offline, properly licensed evidence; never publish access credentials, private receipts or raw market DBNs. Peer forecasts need attributable source records and challenge matching. Negative results are legitimate research outcomes.

**Legacy evidence exception:** The first forecast was accepted through an earlier Notebook 10 workflow; its original native artifacts are missing. We import its scored server response only as labeled recovered evidence, cite the official public Arena challenge and show the native-only sensitivity result. The original model-byte fingerprint for that specific first forecast is not automatically proven by the model currently on disk; verify from the old Notebook 10 if possible. The server timestamp in that response lacks a UTC offset; never assert time-zone-level precision that the evidence does not provide.

**Evidence limitation:** a SHA-256 computed today identifies present file bytes but cannot independently prove when those bytes were first created. For external verification, retain the original API receipt/server timestamp, original Git-ignored evidence and matched Arena settlement pages.


In [ ]:
accepted_n=int(ledger.loc[(ledger.settlement_date>=COHORT_START)&ledger.accepted_with_evidence].shape[0]) if not ledger.empty else 0
settled_n=int(full.eligible_for_metrics.sum()) if not full.empty else 0
first_challenge_evidenced=(not ledger.empty and ORIGINAL_FIRST_CHALLENGE_ID in set(ledger.loc[ledger.accepted_with_evidence,"challenge_id"]))
model_verified=all(x.is_file() for x in [model_file,manifest_file,metadata_file])
all_cohort_deadlines_verified=(not full.empty and len(full.loc[full.in_first_30_accepted])>=COHORT_TARGET_N
    and full.loc[full.in_first_30_accepted,"submission_deadline_status"].isin(["VERIFIED","SERVER_ACCEPTED_SCORING_ELIGIBLE_UNZONED_TIME"]).all())
quality_report={
 "planned_accepted_cohort":COHORT_TARGET_N,
 "native_forecasts_found":len(forecast_paths),
 "native_forecasts_parsed":len(forecasts),
 "unrecognized_forecast_files":len(forecast_errors),
 "native_receipts_found":len(receipt_paths),
 "recovered_legacy_records":len(legacy_rows),
 "original_oct20_forecast_evidenced":bool(first_challenge_evidenced),
 "legacy_native_forecast_receipt_missing":sum(not x.get("native_artifacts_present",False) for x in legacy_rows),
 "legacy_model_byte_provenance_unconfirmed":sum(not x.get("original_model_sha_verified",False) for x in legacy_rows),
 "unrecognized_receipt_files":len(receipt_errors),
 "unrecognized_legacy_files":len(legacy_errors),
 "accepted_native_plus_recovered_evidence_since_start":accepted_n,
 "eligible_confirmed_and_officially_settled":settled_n,
 "official_result_rows_provided":len(arena),
 "peer_rows_provided":len(peer),
 "model_sha256_expected":FROZEN_MODEL_SHA256,
 "local_deployment_verified":model_verified,
 "all_cohort_deadlines_verified":bool(all_cohort_deadlines_verified),
 "analysis_status":"FINAL_READY" if first_challenge_evidenced and model_verified and all_cohort_deadlines_verified and accepted_n>=COHORT_TARGET_N and settled_n==COHORT_TARGET_N and not forecast_errors and not receipt_errors and not legacy_errors else "PRELIMINARY_OR_INCOMPLETE",
}
display(pd.DataFrame(quality_report.items(),columns=["check","result"]))
if not calendar_raw.empty:
    print("Declared non-submissions / market closures / outages:")
    display(calendar_raw.sort_values("settlement_date"))
if quality_report["analysis_status"]!="FINAL_READY":
    display(Markdown("**PRELIMINARY / INCOMPLETE.** Do not label a short-run result as final. Missing proof is *not* evidence of success or failure."))
else:
    print("Primary completeness gate passed; additionally confirm provenance, peer rights and publication review before finalizing.")

## 14 · Reproducible research summary (Markdown-ready, descriptive only)

This notebook is **not a submission client**. It doesn't change the model or the official Arena records. To reproduce, use the original frozen deployment, verified native JSON/receipt archive PLUS explicitly labeled recovered first-notebook console record, populated official-outcome CSV with source URLs, and optional peers/training counts CSV.

**Versioning:** commit this notebook with *no private execution outputs*. Keep raw evidence and populated manual CSVs local or in a properly access-controlled archive. If analysis definitions change, document a new prospective protocol version instead of rewriting the observed history.

**Next components:** Notebook 13 = prospective paper-trading return mechanics, using executions *after* forecast availability; Notebook 14 = volatility/geopolitical-event sensitivity analysis. No ROI is implied by Brier score or classification accuracy.

**Formal interpretation:** A probability forecast can be uncertain yet calibrated, or confident yet wrong. Accuracy, maximum probability, log loss, multiclass Brier, and matched Arena performance each answer different questions. A small cohort and changing market regimes impose severe inference limits.

In [ ]:
if MODEL_METRICS is None:
    statement="No accepted-evidence and officially settled ES forecasts are available in the provided local evidence. Research metrics have not been calculated."
else:
    statement=(f"PRELIMINARY ES study: {MODEL_METRICS['n']}/{COHORT_TARGET_N} officially settled records with native-or-recovered acceptance evidence; "
               f"accuracy {MODEL_METRICS['accuracy']:.1%}; average confidence {MODEL_METRICS['avg_confidence']:.1%}; "
               f"unscaled multiclass Brier {MODEL_METRICS['brier']:.4f}; log loss {MODEL_METRICS['log_loss']:.4f}. "
               "Small sample, conditional on available verified records; do not infer demonstrated predictive skill.")
print(statement)
print("Next: preserve every immutable forecast + receipt, verify official outcomes, and complete the planned 30 forecasts without retraining.")